In [ ]:
from google.colab import files
import pandas as pd
import io

uploaded = files.upload()

filename = list(uploaded.keys())[0]

white_df = pd.read_csv(
    io.BytesIO(uploaded[filename]),
    sep=";"
)

print("WHITE WINE DATASET")
print("--------------------")
print("File:", filename)
print("Shape:", white_df.shape)
print("Duplicates:", white_df.duplicated().sum())
print("Missing values:", white_df.isnull().sum().sum())

print("\nQuality distribution:")
print(white_df["quality"].value_counts().sort_index())

print("\nColumns:")
print(white_df.columns.tolist())

Saving winequality-white.csv to winequality-white.csv
WHITE WINE DATASET
--------------------
File: winequality-white.csv
Shape: (3961, 12)
Duplicates: 0
Missing values: 0

Quality distribution:
quality
3      20
4     153
5    1175
6    1788
7     689
8     131
9       5
Name: count, dtype: int64

Columns:
['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']


In [ ]:
# Features and target
X_white = white_df.drop("quality", axis=1)
y_white = white_df["quality"]

print("X shape:", X_white.shape)
print("y shape:", y_white.shape)

print("\nFeatures:")
print(X_white.columns.tolist())

X shape: (3961, 11)
y shape: (3961,)

Features:
['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']


In [ ]:
from sklearn.model_selection import RepeatedKFold

cv_white = RepeatedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=42
)

print("Total white-wine evaluation folds:",
      cv_white.get_n_splits())

Total white-wine evaluation folds: 15


Step 2 — Mean predictor baseline

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer
from scipy.stats import spearmanr
import numpy as np

def spearman_score(y_true, y_pred):
    result = spearmanr(y_true, y_pred).statistic

    if np.isnan(result):
        return np.nan

    return result

scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2",
    "Spearman": make_scorer(spearman_score)
}

baseline_white = DummyRegressor(strategy="mean")

baseline_white_results = cross_validate(
    baseline_white,
    X_white,
    y_white,
    cv=cv_white,
    scoring=scoring,
    n_jobs=-1
)

white_baseline_mae = -baseline_white_results["test_MAE"]
white_baseline_rmse = -baseline_white_results["test_RMSE"]
white_baseline_r2 = baseline_white_results["test_R2"]

print("WHITE WINE — MEAN PREDICTOR BASELINE")
print("--------------------------------------")
print(
    f"MAE:  {white_baseline_mae.mean():.4f} ± "
    f"{white_baseline_mae.std():.4f}"
)
print(
    f"RMSE: {white_baseline_rmse.mean():.4f} ± "
    f"{white_baseline_rmse.std():.4f}"
)
print(
    f"R²:   {white_baseline_r2.mean():.4f} ± "
    f"{white_baseline_r2.std():.4f}"
)

print("Spearman: N/A (constant mean prediction)")

WHITE WINE — MEAN PREDICTOR BASELINE
--------------------------------------
MAE:  0.6796 ± 0.0202
RMSE: 0.8905 ± 0.0223
R²:   -0.0017 ± 0.0017
Spearman: N/A (constant mean prediction)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, cross_validate, KFold

# Ridge pipeline
ridge_white_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

# Same hyperparameter grid used for red wine
ridge_white_params = {
    "ridge__alpha": [0.01, 0.1, 1, 10, 100]
}

# Inner CV for hyperparameter tuning
inner_cv_white = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

ridge_white_search = GridSearchCV(
    estimator=ridge_white_pipeline,
    param_grid=ridge_white_params,
    scoring="neg_root_mean_squared_error",
    cv=inner_cv_white,
    n_jobs=-1
)

# Outer 5-fold × 3 repeated evaluation
ridge_white_results = cross_validate(
    ridge_white_search,
    X_white,
    y_white,
    cv=cv_white,
    scoring=scoring,
    return_estimator=True,
    n_jobs=-1
)

# Extract metrics
white_ridge_mae = -ridge_white_results["test_MAE"]
white_ridge_rmse = -ridge_white_results["test_RMSE"]
white_ridge_r2 = ridge_white_results["test_R2"]
white_ridge_spearman = ridge_white_results["test_Spearman"]

print("WHITE WINE — RIDGE REGRESSION")
print("--------------------------------")
print(f"MAE:      {white_ridge_mae.mean():.4f} ± {white_ridge_mae.std():.4f}")
print(f"RMSE:     {white_ridge_rmse.mean():.4f} ± {white_ridge_rmse.std():.4f}")
print(f"R²:       {white_ridge_r2.mean():.4f} ± {white_ridge_r2.std():.4f}")
print(f"Spearman: {white_ridge_spearman.mean():.4f} ± {white_ridge_spearman.std():.4f}")

print("\nBest alpha selected in each outer fold:")

best_white_ridge_alphas = [
    est.best_params_["ridge__alpha"]
    for est in ridge_white_results["estimator"]
]

print(best_white_ridge_alphas)

WHITE WINE — RIDGE REGRESSION
--------------------------------
MAE:      0.5810 ± 0.0149
RMSE:     0.7521 ± 0.0221
R²:       0.2851 ± 0.0286
Spearman: 0.5626 ± 0.0168

Best alpha selected in each outer fold:
[100, 100, 100, 100, 1, 100, 100, 10, 1, 100, 100, 100, 1, 100, 100]


Next — White Wine KNN

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, cross_validate

knn_white_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor())
])

# Same search space used for red wine
knn_white_params = {
    "knn__n_neighbors": [3, 5, 7, 9, 11, 15],
    "knn__weights": ["uniform", "distance"]
}

knn_white_search = GridSearchCV(
    estimator=knn_white_pipeline,
    param_grid=knn_white_params,
    scoring="neg_root_mean_squared_error",
    cv=inner_cv_white,
    n_jobs=-1
)

knn_white_results = cross_validate(
    knn_white_search,
    X_white,
    y_white,
    cv=cv_white,
    scoring=scoring,
    return_estimator=True,
    n_jobs=-1
)

white_knn_mae = -knn_white_results["test_MAE"]
white_knn_rmse = -knn_white_results["test_RMSE"]
white_knn_r2 = knn_white_results["test_R2"]
white_knn_spearman = knn_white_results["test_Spearman"]

print("WHITE WINE — KNN REGRESSOR")
print("--------------------------------")
print(f"MAE:      {white_knn_mae.mean():.4f} ± {white_knn_mae.std():.4f}")
print(f"RMSE:     {white_knn_rmse.mean():.4f} ± {white_knn_rmse.std():.4f}")
print(f"R²:       {white_knn_r2.mean():.4f} ± {white_knn_r2.std():.4f}")
print(f"Spearman: {white_knn_spearman.mean():.4f} ± {white_knn_spearman.std():.4f}")

print("\nBest KNN parameters selected in each outer fold:")

for i, est in enumerate(knn_white_results["estimator"], 1):
    print(f"Fold {i}: {est.best_params_}")

WHITE WINE — KNN REGRESSOR
--------------------------------
MAE:      0.5549 ± 0.0164
RMSE:     0.7209 ± 0.0232
R²:       0.3434 ± 0.0236
Spearman: 0.6012 ± 0.0152

Best KNN parameters selected in each outer fold:
Fold 1: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Fold 2: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Fold 3: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Fold 4: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Fold 5: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Fold 6: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Fold 7: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Fold 8: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Fold 9: {'knn__n_neighbors': 11, 'knn__weights': 'distance'}
Fold 10: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Fold 11: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Fold 12: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Fold 13: {'knn__n_neighbors': 15, 'knn__weights': '

Next — White Wine Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, cross_validate

rf_white = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Reduced tuning grid
rf_white_params = {
    "max_depth": [None, 10],
    "min_samples_leaf": [1, 2]
}

# Small inner CV to keep computation manageable
rf_white_search = GridSearchCV(
    estimator=rf_white,
    param_grid=rf_white_params,
    scoring="neg_root_mean_squared_error",
    cv=2,
    n_jobs=1
)

# Outer evaluation remains 5-fold × 3 repeats
rf_white_results = cross_validate(
    rf_white_search,
    X_white,
    y_white,
    cv=cv_white,
    scoring=scoring,
    return_estimator=True,
    n_jobs=1
)

# Metrics
white_rf_mae = -rf_white_results["test_MAE"]
white_rf_rmse = -rf_white_results["test_RMSE"]
white_rf_r2 = rf_white_results["test_R2"]
white_rf_spearman = rf_white_results["test_Spearman"]

print("WHITE WINE — RANDOM FOREST REGRESSOR")
print("--------------------------------------")
print(f"MAE:      {white_rf_mae.mean():.4f} ± {white_rf_mae.std():.4f}")
print(f"RMSE:     {white_rf_rmse.mean():.4f} ± {white_rf_rmse.std():.4f}")
print(f"R²:       {white_rf_r2.mean():.4f} ± {white_rf_r2.std():.4f}")
print(f"Spearman: {white_rf_spearman.mean():.4f} ± {white_rf_spearman.std():.4f}")

print("\nBest Random Forest parameters selected in each outer fold:")

for i, est in enumerate(rf_white_results["estimator"], 1):
    print(f"Fold {i}: {est.best_params_}")

WHITE WINE — RANDOM FOREST REGRESSOR
--------------------------------------
MAE:      0.5516 ± 0.0142
RMSE:     0.7094 ± 0.0222
R²:       0.3641 ± 0.0249
Spearman: 0.6189 ± 0.0200

Best Random Forest parameters selected in each outer fold:
Fold 1: {'max_depth': 10, 'min_samples_leaf': 2}
Fold 2: {'max_depth': 10, 'min_samples_leaf': 2}
Fold 3: {'max_depth': 10, 'min_samples_leaf': 2}
Fold 4: {'max_depth': 10, 'min_samples_leaf': 2}
Fold 5: {'max_depth': 10, 'min_samples_leaf': 2}
Fold 6: {'max_depth': 10, 'min_samples_leaf': 2}
Fold 7: {'max_depth': 10, 'min_samples_leaf': 2}
Fold 8: {'max_depth': 10, 'min_samples_leaf': 1}
Fold 9: {'max_depth': 10, 'min_samples_leaf': 1}
Fold 10: {'max_depth': 10, 'min_samples_leaf': 2}
Fold 11: {'max_depth': 10, 'min_samples_leaf': 2}
Fold 12: {'max_depth': 10, 'min_samples_leaf': 2}
Fold 13: {'max_depth': 10, 'min_samples_leaf': 2}
Fold 14: {'max_depth': 10, 'min_samples_leaf': 2}
Fold 15: {'max_depth': 10, 'min_samples_leaf': 2}


Final white-wine model — SVR

In [ ]:
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, cross_validate

svr_white_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(kernel="rbf"))
])

svr_white_params = {
    "svr__C": [1, 10],
    "svr__epsilon": [0.1, 0.2]
}

svr_white_search = GridSearchCV(
    estimator=svr_white_pipeline,
    param_grid=svr_white_params,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1
)

svr_white_results = cross_validate(
    svr_white_search,
    X_white,
    y_white,
    cv=cv_white,
    scoring=scoring,
    return_estimator=True,
    n_jobs=-1
)

white_svr_mae = -svr_white_results["test_MAE"]
white_svr_rmse = -svr_white_results["test_RMSE"]
white_svr_r2 = svr_white_results["test_R2"]
white_svr_spearman = svr_white_results["test_Spearman"]

print("WHITE WINE — SVR")
print("--------------------------------")
print(f"MAE:      {white_svr_mae.mean():.4f} ± {white_svr_mae.std():.4f}")
print(f"RMSE:     {white_svr_rmse.mean():.4f} ± {white_svr_rmse.std():.4f}")
print(f"R²:       {white_svr_r2.mean():.4f} ± {white_svr_r2.std():.4f}")
print(f"Spearman: {white_svr_spearman.mean():.4f} ± {white_svr_spearman.std():.4f}")

print("\nBest SVR parameters selected in each outer fold:")

for i, est in enumerate(svr_white_results["estimator"], 1):
    print(f"Fold {i}: {est.best_params_}")

WHITE WINE — SVR
--------------------------------
MAE:      0.5434 ± 0.0165
RMSE:     0.7080 ± 0.0219
R²:       0.3664 ± 0.0293
Spearman: 0.6207 ± 0.0218

Best SVR parameters selected in each outer fold:
Fold 1: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 2: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 3: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 4: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 5: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 6: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 7: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 8: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 9: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 10: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 11: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 12: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 13: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 14: {'svr__C': 1, 'svr__epsilon': 0.1}
Fold 15: {'svr__C': 1, 'svr__epsilon': 0.2}


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import friedmanchisquare, wilcoxon
from itertools import combinations

# ============================================================
# 1. SAVE WHITE-WINE FOLD-LEVEL RESULTS
# ============================================================

white_fold_results = pd.DataFrame({
    "Fold": np.arange(1, len(white_ridge_mae) + 1),

    "Ridge_MAE": white_ridge_mae,
    "Ridge_RMSE": white_ridge_rmse,
    "Ridge_R2": white_ridge_r2,
    "Ridge_Spearman": white_ridge_spearman,

    "KNN_MAE": white_knn_mae,
    "KNN_RMSE": white_knn_rmse,
    "KNN_R2": white_knn_r2,
    "KNN_Spearman": white_knn_spearman,

    "RF_MAE": white_rf_mae,
    "RF_RMSE": white_rf_rmse,
    "RF_R2": white_rf_r2,
    "RF_Spearman": white_rf_spearman,

    "SVR_MAE": white_svr_mae,
    "SVR_RMSE": white_svr_rmse,
    "SVR_R2": white_svr_r2,
    "SVR_Spearman": white_svr_spearman
})

white_fold_results.to_csv(
    "white_wine_fold_results.csv",
    index=False
)

print("WHITE-WINE FOLD RESULTS")
display(white_fold_results)


# ============================================================
# 2. FRIEDMAN TEST ON RMSE
# ============================================================

friedman_stat, friedman_p = friedmanchisquare(
    white_ridge_rmse,
    white_knn_rmse,
    white_rf_rmse,
    white_svr_rmse
)

print("\nFRIEDMAN TEST — WHITE WINE RMSE")
print("--------------------------------")
print(f"Statistic: {friedman_stat:.4f}")
print(f"p-value:   {friedman_p:.6f}")

if friedman_p < 0.05:
    print("Result: Significant differences exist among the algorithms.")
else:
    print("Result: No statistically significant difference was detected.")


# ============================================================
# 3. PAIRWISE WILCOXON TESTS
# ============================================================

models_white = {
    "Ridge": white_ridge_rmse,
    "KNN": white_knn_rmse,
    "Random Forest": white_rf_rmse,
    "SVR": white_svr_rmse
}

pairwise_white = []

for model1, model2 in combinations(models_white.keys(), 2):

    stat, p = wilcoxon(
        models_white[model1],
        models_white[model2],
        alternative="two-sided"
    )

    pairwise_white.append({
        "Model 1": model1,
        "Model 2": model2,
        "Wilcoxon Statistic": stat,
        "Raw p-value": p
    })

white_pairwise_df = pd.DataFrame(pairwise_white)


# ============================================================
# 4. HOLM MULTIPLE-COMPARISON CORRECTION
# ============================================================

m = len(white_pairwise_df)

sorted_indices = white_pairwise_df[
    "Raw p-value"
].argsort()

sorted_p = white_pairwise_df.loc[
    sorted_indices,
    "Raw p-value"
].values

holm_adjusted = []

for i, p in enumerate(sorted_p):
    adjusted_p = min((m - i) * p, 1.0)
    holm_adjusted.append(adjusted_p)

# Ensure monotonic Holm-adjusted p-values
for i in range(1, len(holm_adjusted)):
    holm_adjusted[i] = max(
        holm_adjusted[i],
        holm_adjusted[i - 1]
    )

white_pairwise_df.loc[
    sorted_indices,
    "Holm Adjusted p-value"
] = holm_adjusted

white_pairwise_df["Significant (α=0.05)"] = (
    white_pairwise_df[
        "Holm Adjusted p-value"
    ] < 0.05
)

print("\nWILCOXON POST-HOC TEST — WHITE WINE")
display(white_pairwise_df)

white_pairwise_df.to_csv(
    "white_wine_wilcoxon_posthoc.csv",
    index=False
)


# ============================================================
# 5. COMPUTATIONAL COST
# ============================================================

white_cost_results = pd.DataFrame({
    "Model": [
        "Ridge",
        "KNN",
        "Random Forest",
        "SVR"
    ],

    "Mean Fit Time (s)": [
        ridge_white_results["fit_time"].mean(),
        knn_white_results["fit_time"].mean(),
        rf_white_results["fit_time"].mean(),
        svr_white_results["fit_time"].mean()
    ],

    "SD Fit Time (s)": [
        ridge_white_results["fit_time"].std(),
        knn_white_results["fit_time"].std(),
        rf_white_results["fit_time"].std(),
        svr_white_results["fit_time"].std()
    ],

    "Mean Score Time (s)": [
        ridge_white_results["score_time"].mean(),
        knn_white_results["score_time"].mean(),
        rf_white_results["score_time"].mean(),
        svr_white_results["score_time"].mean()
    ]
})

print("\nWHITE WINE — COMPUTATIONAL COST")
display(white_cost_results)

white_cost_results.to_csv(
    "white_wine_computational_cost.csv",
    index=False
)

print("\nAll white-wine result files saved successfully.")

WHITE-WINE FOLD RESULTS


,Fold,Ridge_MAE,Ridge_RMSE,Ridge_R2,Ridge_Spearman,KNN_MAE,KNN_RMSE,KNN_R2,KNN_Spearman,RF_MAE,RF_RMSE,RF_R2,RF_Spearman,SVR_MAE,SVR_RMSE,SVR_R2,SVR_Spearman
0,1,0.599992,0.782093,0.254178,0.540835,0.579274,0.756082,0.302962,0.582174,0.571745,0.741415,0.329743,0.602241,0.570887,0.742108,0.328490,0.606661
1,2,0.584817,0.751322,0.300443,0.570024,0.578828,0.741595,0.318440,0.588556,0.570509,0.733390,0.333438,0.602826,0.560679,0.725063,0.348489,0.614074
2,3,0.557531,0.730334,0.266488,0.548476,0.529527,0.696275,0.333307,0.591125,0.532770,0.691548,0.342330,0.604427,0.527365,0.695275,0.335221,0.600649
3,4,0.583106,0.757349,0.316083,0.583057,0.550987,0.720588,0.380865,0.625554,0.543934,0.704478,0.408239,0.661063,0.526116,0.691762,0.429409,0.663526
4,5,0.579579,0.741688,0.280840,0.567229,0.548361,0.701825,0.356068,0.605122,0.541726,0.684648,0.387203,0.620711,0.535584,0.688265,0.380711,0.621568
5,6,0.597987,0.774300,0.297068,0.565395,0.564767,0.734639,0.367235,0.626452,0.567250,0.737398,0.362472,0.613295,0.547810,0.721799,0.389160,0.638246
6,7,0.557305,0.731299,0.311110,0.581245,0.539982,0.713060,0.345044,0.598895,0.542267,0.697425,0.373450,0.627209,0.530571,0.696531,0.375055,0.628903
7,8,0.555948,0.707059,0.303281,0.567038,0.527609,0.670069,0.374272,0.615470,0.526983,0.660630,0.391777,0.630023,0.511185,0.651389,0.408675,0.648612
8,9,0.601249,0.792657,0.254284,0.569383,0.571345,0.756934,0.319983,0.593653,0.562344,0.728803,0.369590,0.637033,0.542071,0.722681,0.380136,0.640735
9,10,0.588304,0.749798,0.273960,0.537211,0.566909,0.733382,0.305402,0.572138,0.555114,0.717023,0.336045,0.595365,0.562653,0.724468,0.322185,0.582819



FRIEDMAN TEST — WHITE WINE RMSE
--------------------------------
Statistic: 33.0800
p-value:   0.000000
Result: Significant differences exist among the algorithms.

WILCOXON POST-HOC TEST — WHITE WINE


,Model 1,Model 2,Wilcoxon Statistic,Raw p-value,Holm Adjusted p-value,Significant (α=0.05)
0,Ridge,KNN,0.0,0.000061,0.000366,True
1,Ridge,Random Forest,0.0,0.000061,0.000366,True
2,Ridge,SVR,0.0,0.000061,0.000366,True
3,KNN,Random Forest,7.0,0.001160,0.002319,True
4,KNN,SVR,5.0,0.000610,0.001831,True
5,Random Forest,SVR,48.0,0.524475,0.524475,False



WHITE WINE — COMPUTATIONAL COST


,Model,Mean Fit Time (s),SD Fit Time (s),Mean Score Time (s)
0,Ridge,0.387888,0.074537,0.011180
1,KNN,5.212899,1.290547,0.115229
2,Random Forest,9.531665,0.517317,0.049191
3,SVR,13.364881,1.123953,0.301284



All white-wine result files saved successfully.


NameError: name 'combined_results' is not defined